# Jersey colour classification in football video


In [1]:
import torch
import pandas as pd
import numpy as np


from dataset import load_manifest, get_loader
from models import load_model
from extract_embeddings import get_embeddings, extract_all_models
from classification_clustering import compare_models, evaluate_single_method, run_clustering
from visualization import interactive_embedding_view, plot_confusion_matrix_clustering

d:\Anaconda\envs\footpass\Lib\site-packages\torchreid\reid\metrics\rank.py:11: UserWarning: Cython evaluation (very fast so highly recommended) is unavailable, now use python evaluation.
  warnings.warn(


In [2]:
import warnings
warnings.filterwarnings("ignore", module='umap')

device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
manifest_path = "dataset_v1/manifest_with_splits.csv"

df = load_manifest(manifest_path)

print("dataset size:", len(df))
df.head()

dataset size: 64291


,crop_path,label,game,src_image,x1,y1,x2,y2,frame_idx,player_id,role_name,left2right,split
0,dataset_v1/crops/game_10_H1__frame_001877__i0_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1399,522,1475,667,1877,101,Left Central Back,1,train
1,dataset_v1/crops/game_10_H1__frame_001877__i1_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1524,290,1591,408,1877,103,Right Central Back,1,train
2,dataset_v1/crops/game_10_H1__frame_001877__i2_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1091,874,1171,1067,1877,104,Left Back,1,train
3,dataset_v1/crops/game_10_H1__frame_001877__i3_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,77,549,126,704,1877,105,Left Midfielder,1,train
4,dataset_v1/crops/game_10_H1__frame_001877__i4_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,185,169,223,269,1877,106,Right Winger,1,train


In [4]:
df["game"].unique()[:20]

<StringArray>
['game_10_H1', 'game_10_H2', 'game_11_H1', 'game_11_H2', 'game_12_H1',
 'game_12_H2', 'game_13_H1', 'game_13_H2', 'game_14_H1', 'game_14_H2',
 'game_15_H1', 'game_15_H2', 'game_16_H1', 'game_16_H2', 'game_17_H1',
 'game_17_H2', 'game_19_H1', 'game_19_H2',  'game_1_H1',  'game_1_H2']
Length: 20, dtype: str

## game_30


In [5]:
base_game = "game_30"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 798 H2: 713 Total: 1511


In [6]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_left     759
team_right    709
goalkeeper     43
Name: count, dtype: int64


In [7]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["osnet", "dino"],
)


  модель: osnet


d:\Anaconda\envs\footpass\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)


Successfully loaded imagenet pretrained weights from "C:\Users\Антон/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
loading cached embeddings
  готово: shape=(1511, 512)

  модель: dino


d:\Anaconda\envs\footpass\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


loading cached embeddings
  готово: shape=(1511, 768)


### Compare methods

In [8]:
method = 'log_reg'
df_log_reg = evaluate_single_method(embeddings, method)
df_log_reg

evaluating osnet for log_reg...
evaluating dino for log_reg...


,accuracy,macro_f1
model,,
osnet,0.9637,0.8528
dino,0.9769,0.9448


In [9]:
method = 'mlp'
df_mlp = evaluate_single_method(embeddings, method)
df_mlp

evaluating osnet for mlp...
evaluating dino for mlp...


,accuracy,macro_f1
model,,
osnet,0.9736,0.9425
dino,0.9769,0.9448


In [10]:
method = 'kmeans'
df_kmeans = evaluate_single_method(embeddings, method)
df_kmeans

evaluating osnet for kmeans...
evaluating dino for kmeans...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.7095,0.573,0.7637,0.6295,0.7445,0.6142,0.8140,0.5871
dino,0.7664,0.626,0.9775,0.9522,0.9808,0.9793,0.7366,0.5447


In [11]:
method = 'hdbscan'
df_hdbscan = evaluate_single_method(embeddings, method)
df_hdbscan

evaluating osnet for hdbscan...
evaluating dino for hdbscan...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,NaN,NaN,0.9755,0.9509,0.9775,0.9567,0.9768,0.9527
dino,0.1509,0.1509,0.9775,0.9522,0.9815,0.9798,0.9801,0.9789


In [12]:
method = 'gmm'
df_gmm = evaluate_single_method(embeddings, method)
df_gmm

evaluating osnet for gmm...
evaluating dino for gmm...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.7095,0.573,0.9285,0.8038,0.7551,0.5556,0.7935,0.5761
dino,0.7664,0.626,0.9775,0.9522,0.7968,0.6533,0.8114,0.5901


### Visualization

In [13]:
df_viz = df_match[["crop_path", "game", "frame_idx", "player_id"]].reset_index(drop=True)

name = "dino"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1511 точек...
Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_30_H1', '1093', '201',
  …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_30_H1', '1093', '201',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCABQADwDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDPxS4p4UDml2ivibnmpERoqXaKYyndxS3GJx7/AJ

In [14]:
model_name = "dino"
method = 'kmeans'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=False, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,747,12,0
team_right,14,695,0
goalkeeper,1,7,35


In [15]:
model_name = "dino"
method = 'hdbscan'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=True, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,744,15,0
team_right,13,696,0
goalkeeper,1,1,41


In [17]:
model_name = "dino"
method = 'gmm'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=False, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,747,12,0
team_right,14,695,0
goalkeeper,1,7,35


## game_40


In [40]:
base_game = "game_40"
df_h1 = df[df["game"] == f"{base_game}_H1"].copy()
df_h2 = df[df["game"] == f"{base_game}_H2"].copy()

print("H1:", len(df_h1), "H2:", len(df_h2), "Total:", len(df_h1) + len(df_h2))

H1: 740 H2: 712 Total: 1452


In [41]:
def flip_lr_label(label: str) -> str:
    if label == "team_left":
        return "team_right"
    if label == "team_right":
        return "team_left"
    return label

df_h2["label"] = df_h2["label"].map(flip_lr_label)

df_match = pd.concat([df_h1, df_h2], ignore_index=True)

print(df_match["label"].value_counts())

label
team_left     705
team_right    694
goalkeeper     53
Name: count, dtype: int64


In [42]:
embeddings = extract_all_models(
    df_match=df_match,
    game_id=base_game,
    device=device,
    model_names=["osnet", "dino", "clip"],
)


  модель: osnet
Successfully loaded imagenet pretrained weights from "C:\Users\Антон/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"
loading cached embeddings
  готово: shape=(1452, 512)

  модель: dino


d:\Anaconda\envs\footpass\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)
d:\Anacond

loading cached embeddings
  готово: shape=(1452, 768)

  модель: clip
loading cached embeddings
  готово: shape=(1452, 512)


### Metrics

In [20]:
method = 'log_reg'
df_log_reg = evaluate_single_method(embeddings, method)
df_log_reg

evaluating osnet for log_reg...
evaluating dino for log_reg...
evaluating clip for log_reg...


,accuracy,macro_f1
model,,
osnet,0.9691,0.9477
dino,0.9725,0.9320
clip,0.9588,0.7060


In [21]:
method = 'mlp'
df_mlp = evaluate_single_method(embeddings, method)
df_mlp

evaluating osnet for mlp...
evaluating dino for mlp...
evaluating clip for mlp...


,accuracy,macro_f1
model,,
osnet,0.9553,0.8221
dino,0.9759,0.9524
clip,0.9863,0.9758


In [22]:
method = 'kmeans'
df_kmeans = evaluate_single_method(embeddings, method)
df_kmeans

evaluating osnet for kmeans...
evaluating dino for kmeans...
evaluating clip for kmeans...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.7479,0.5677,0.9174,0.7976,0.7231,0.5449,0.7762,0.6488
dino,0.7431,0.6061,0.7707,0.5677,0.9821,0.9786,0.7417,0.5516
clip,0.7342,0.5933,0.8492,0.7149,0.7638,0.6393,0.7576,0.5603


In [23]:
method = 'hdbscan'
df_hdbscan = evaluate_single_method(embeddings, method)
df_hdbscan

evaluating osnet for hdbscan...
evaluating dino for hdbscan...
evaluating clip for hdbscan...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.1123,0.1123,0.9449,0.6420,0.9415,0.6396,0.9421,0.6401
dino,NaN,NaN,0.9807,0.9746,0.9835,0.9796,0.9814,0.9750
clip,NaN,NaN,0.9835,0.9796,0.9821,0.9755,0.9821,0.9755


In [24]:
method = 'gmm'
df_gmm = evaluate_single_method(embeddings, method)
df_gmm

evaluating osnet for gmm...
evaluating dino for gmm...
evaluating clip for gmm...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.7479,0.5677,0.9483,0.8557,0.8843,0.7555,0.7727,0.5687
dino,0.7431,0.6061,0.7879,0.5766,0.9828,0.9791,0.7466,0.5547
clip,0.7342,0.5933,0.8712,0.7375,0.8430,0.7090,0.9594,0.8859


### Visualization

In [25]:
df_viz = df_match[["crop_path", "game", "frame_idx", "player_id"]].reset_index(drop=True)

name = "dino"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=None,
    preload_images=True,
)


Снижение размерности (UMAP) для 1452 точек...
Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
   …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_40_H1', '162', '202',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCAAiABYDASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwDY1uzkvbDbBrA0uZXDK7FfnAByvzcf/qpYZImtYzH

In [43]:
model_name = "dino"
method = 'kmeans'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=True, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,685,20,0
team_right,4,690,0
goalkeeper,1,2,50


In [44]:
model_name = "dino"
method = 'hdbscan'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=True, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,687,18,0
team_right,7,687,0
goalkeeper,1,2,50


In [45]:
model_name = "dino"
method = 'gmm'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=True, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,687,11,7
team_right,5,680,9
goalkeeper,1,0,52


# Video

In [15]:
# from pathlib import Path
# from make_dataset import build_dataset_for_game_video

# game_dir = Path("./output_clips/game_1_H1")
# out_dir = Path("./dataset_video_v1/crops")
# out_dir.mkdir(parents=True, exist_ok=True)

# df, bad, csv_path, video_path = build_dataset_for_game_video(
#     game_dir,
#     out_dir,
#     video_path=Path("./output_clips/game_1_H1/game_1_H1.mp4"),
#     players_csv=Path("./output_clips/game_1_H1/players.csv"),
#     min_wh=20,
#     jpeg_quality=90,
#     pad=6,
#     frame_stride=2
# )

# df.head()

In [16]:
# from pathlib import Path

# out_root = Path("./dataset_video_v1")
# out_root.mkdir(parents=True, exist_ok=True)
# df.to_csv(out_root / "manifest.csv", index=False)

In [68]:
manifest_path = "dataset_video_v1/manifest.csv"

df = load_manifest(manifest_path)

print("dataset size:", len(df))
print(df["label"].value_counts())
# df.head()

dataset size: 10157
label
team_right    4973
team_left     4840
goalkeeper     344
Name: count, dtype: int64


In [69]:
embeddings = extract_all_models(
    df_match=df,
    game_id=None,
    device=device,
    model_names=["osnet", "dino"],
)


  модель: osnet
Successfully loaded imagenet pretrained weights from "C:\Users\Антон/.cache\torch\checkpoints\osnet_x1_0_imagenet.pth"


d:\Anaconda\envs\footpass\Lib\site-packages\torchreid\reid\models\osnet.py:482: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(cached_file)


loading cached embeddings
  готово: shape=(10157, 512)

  модель: dino


d:\Anaconda\envs\footpass\Lib\site-packages\timm\models\_factory.py:138: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


loading cached embeddings
  готово: shape=(10157, 768)


In [70]:
method = 'log_reg'
df_log_reg = evaluate_single_method(embeddings, method)
df_log_reg

evaluating osnet for log_reg...
evaluating dino for log_reg...


,accuracy,macro_f1
model,,
osnet,0.9818,0.9829
dino,0.9847,0.9895


In [71]:
method = 'mlp'
df_mlp = evaluate_single_method(embeddings, method)
df_mlp

evaluating osnet for mlp...
evaluating dino for mlp...


,accuracy,macro_f1
model,,
osnet,0.9833,0.9885
dino,0.9877,0.9915


In [72]:
method = 'kmeans'
df_kmeans = evaluate_single_method(embeddings, method)
df_kmeans

evaluating osnet for kmeans...
evaluating dino for kmeans...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.8122,0.6736,0.9602,0.8741,0.8636,0.6437,0.7550,0.6405
dino,0.8216,0.6414,0.8536,0.6042,0.8556,0.6059,0.6462,0.5270


In [73]:
method = 'hdbscan'
df_hdbscan = evaluate_single_method(embeddings, method)
df_hdbscan

evaluating osnet for hdbscan...
evaluating dino for hdbscan...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.2674,0.2567,0.5360,0.0649,0.9538,0.5236,0.5241,0.0548
dino,0.4472,0.4395,0.4389,0.0427,0.1796,0.0237,0.1849,0.0239


In [74]:
method = 'gmm'
df_gmm = evaluate_single_method(embeddings, method)
df_gmm

evaluating osnet for gmm...
evaluating dino for gmm...


,accuracy,macro_f1,accuracy_umap,macro_f1_umap,accuracy_umap_pca,macro_f1_umap_pca,accuracy_umap_pca_scale,macro_f1_umap_pca_scale
model,,,,,,,,
osnet,0.8187,0.6824,0.9602,0.8741,0.8658,0.6452,0.8251,0.6939
dino,0.8222,0.6421,0.8536,0.6042,0.8556,0.6059,0.7091,0.5614


In [75]:
df_viz = df[["crop_path", "game", "frame_idx", "player_id"]].reset_index(drop=True)

name = "osnet"
X, y = embeddings[name]

interactive_embedding_view(
    X=X,
    y=y,
    df=df_viz,
    method="umap",
    sample_n=1500,
    preload_images=True,
)


Снижение размерности (UMAP) для 1500 точек...
Готово.
Кодирование изображений...
Готово.


    'data': [{'customdata': array([['team_left', 'game_1_H1', '80', '101',
     …

FigureWidget({
    'data': [{'customdata': array([['team_left', 'game_1_H1', '80', '101',
                                    '/9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAUDBAQEAwUEBAQFBQUGBwwIBwcHBw8LCwkMEQ8SEhEPERETFhwXExQaFRERGCEYGh0dHx8fExciJCIeJBweHx7/2wBDAQUFBQcGBw4ICA4eFBEUHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh4eHh7/wAARCABVAC8DASIAAhEBAxEB/8QAHwAAAQUBAQEBAQEAAAAAAAAAAAECAwQFBgcICQoL/8QAtRAAAgEDAwIEAwUFBAQAAAF9AQIDAAQRBRIhMUEGE1FhByJxFDKBkaEII0KxwRVS0fAkM2JyggkKFhcYGRolJicoKSo0NTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqDhIWGh4iJipKTlJWWl5iZmqKjpKWmp6ipqrKztLW2t7i5usLDxMXGx8jJytLT1NXW19jZ2uHi4+Tl5ufo6erx8vP09fb3+Pn6/8QAHwEAAwEBAQEBAQEBAQAAAAAAAAECAwQFBgcICQoL/8QAtREAAgECBAQDBAcFBAQAAQJ3AAECAxEEBSExBhJBUQdhcRMiMoEIFEKRobHBCSMzUvAVYnLRChYkNOEl8RcYGRomJygpKjU2Nzg5OkNERUZHSElKU1RVVldYWVpjZGVmZ2hpanN0dXZ3eHl6goOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4uPk5ebn6Onq8vP09fb3+Pn6/9oADAMBAAIRAxEAPwD5kIPcUe9Pkzyaif7jY64OK4XFM4dx28Z4OaQk5zkYr

In [76]:
model_name = "osnet"
method = 'kmeans'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=False, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,4690,150,0
team_right,90,4883,0
goalkeeper,4,160,180


In [77]:
model_name = "osnet"
method = 'hdbscan'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=True, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,4674,147,0
team_right,107,4835,0
goalkeeper,0,96,179


In [78]:
print(np.unique(y_true, return_counts=True))
print(np.unique(clusters, return_counts=True))


(array([0, 1, 2]), array([4840, 4973,  344]))
(array([0, 1, 2, 3, 4]), array([ 179,   31, 5078, 4781,   88]))


In [79]:
model_name = "osnet"
method = 'gmm'
X, y_true = embeddings[model_name]
_, clusters = run_clustering(X, y_true, method, is_umap=True, is_pca=False, is_scale=False)
plot_confusion_matrix_clustering(y_true, clusters)

,team_left,team_right,goalkeeper
team_left,4690,150,0
team_right,90,4883,0
goalkeeper,4,160,180


### Подбор гиперпараметов 

In [27]:
import optuna
import numpy as np

def objective_umap(trial, X, y, method="kmeans"):

    n_components = trial.suggest_int("n_components", 2, 20)
    n_neighbors = trial.suggest_int("n_neighbors", 5, 100)
    min_dist = trial.suggest_float("min_dist", 0.0, 0.8)
    metric = trial.suggest_categorical("metric", ["cosine", "euclidean"])

    seeds = [0, 1, 2]
    scores = []

    for seed in seeds:

        reducer = umap.UMAP(
            n_components=n_components,
            n_neighbors=n_neighbors,
            min_dist=min_dist,
            metric=metric,
            random_state=seed
        )

        X_umap = reducer.fit_transform(l2norm(X))

        results, _ = run_clustering(
            X_umap,
            y,
            method=method,
            is_umap=False
        )

        score = results["macro_f1_cluster"]

        if not np.isnan(score):
            scores.append(score)

    if len(scores) == 0:
        return -1.0

    return float(np.mean(scores))

def print_callback(study, trial):

    print(
        f"trial {trial.number} finished | "
        f"score={trial.value:.4f} | "
        f"best={study.best_value:.4f}"
    )

In [28]:
study = optuna.create_study(direction="maximize")

study.optimize(
    lambda trial: objective_umap(trial, X, y, method="kmeans"),
    n_trials=60,
    show_progress_bar=True,
    callbacks=[print_callback]
)
print("best score:", study.best_value)
print("best params:", study.best_params)

[I 2026-03-11 01:52:44,912] A new study created in memory with name: no-name-a3918482-46a9-426b-b546-1d31963a087a


  0%|          | 0/60 [00:00<?, ?it/s]

d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 01:53:55,906] Trial 0 finished with value: 0.6046166402003917 and parameters: {'n_components': 10, 'n_neighbors': 48, 'min_dist': 0.26620802307552893, 'metric': 'euclidean'}. Best is trial 0 with value: 0.6046166402003917.
trial 0 finished | score=0.6046 | best=0.6046


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

[I 2026-03-11 02:46:01,446] Trial 57 finished with value: 0.6079766254899531 and parameters: {'n_components': 17, 'n_neighbors': 40, 'min_dist': 0.772132640097133, 'metric': 'cosine'}. Best is trial 33 with value: 0.7548119788517086.
trial 57 finished | score=0.6080 | best=0.7548


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 02:46:33,485] Trial 58 finished with value: 0.6040953105089198 and parameters: {'n_components': 19, 'n_neighbors': 53, 'min_dist': 0.672814502966224, 'metric': 'euclidean'}. Best is trial 33 with value: 0.7548119788517086.
trial 58 finished | score=0.6041 | best=0.7548


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 02:46:56,691] Trial 59 finished with value: 0.7005640811265786 and parameters: {'n_components': 10, 'n_neighbors': 35, 'min_dist': 0.799694831511757, 'metric': 'euclidean'}. Best is trial 33 with value: 0.7548119788517086.
trial 59 finished | score=0.7006 | best=0.7548
best score: 0.7548119788517086
best params: {'n_components': 12, 'n_neighbors': 83, 'min_dist': 0.42632588765351775, 'metric': 'euclidean'}


In [13]:
import numpy as np
import optuna
from sklearn.decomposition import PCA
import umap

def l2norm(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(norms, eps, None)


def objective_pca_n_components(trial, X, y, method="kmeans"):
    X_norm = l2norm(X)

    max_comp = min(50, X_norm.shape[1], X_norm.shape[0])
    n_components_pca = trial.suggest_int("pca_n_components", 2, max_comp)

    X_pca = PCA(n_components=n_components_pca).fit_transform(X_norm)

    seeds = [0, 1, 2]
    scores = []

    for seed in seeds:
        reducer = umap.UMAP(
            n_components=10,
            n_neighbors=30,
            min_dist=0.0,
            metric="cosine",
            random_state=seed
        )

        X_umap = reducer.fit_transform(X_pca)

        results, _ = run_clustering(
            X_umap,
            y,
            method=method,
            is_umap=False,
            is_pca=False,
            is_scale=False
        )

        score = results["macro_f1_cluster"]

        if not np.isnan(score):
            scores.append(score)

    if len(scores) == 0:
        return -1.0

    return float(np.mean(scores))

In [17]:
X, y = embeddings["osnet"]
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    lambda trial: objective_pca_n_components(trial, X, y, method="kmeans"),
    n_trials=40,
    show_progress_bar=True
)

print("best score:", study.best_value)
print("best params:", study.best_params)

[I 2026-03-11 08:19:26,809] A new study created in memory with name: no-name-60e7338f-302b-4b4b-93db-3b93092dcb5f


  0%|          | 0/40 [00:00<?, ?it/s]

d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 08:20:15,900] Trial 0 finished with value: 0.7232119366373125 and parameters: {'pca_n_components': 20}. Best is trial 0 with value: 0.7232119366373125.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

[I 2026-03-11 08:47:35,161] Trial 37 finished with value: 0.6786146262825583 and parameters: {'pca_n_components': 9}. Best is trial 3 with value: 0.8483594274375698.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 08:48:21,784] Trial 38 finished with value: 0.6329522168447439 and parameters: {'pca_n_components': 30}. Best is trial 3 with value: 0.8483594274375698.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 08:49:07,006] Trial 39 finished with value: 0.6709407670036184 and parameters: {'pca_n_components': 21}. Best is trial 3 with value: 0.8483594274375698.
best score: 0.8483594274375698
best params: {'pca_n_components': 31}


In [16]:
import numpy as np
import optuna
import umap
import hdbscan

from sklearn.decomposition import PCA
from metrics import macro_f1_clustering


def objective_hdbscan_pca31(trial, X, y):
    # 1) Нормировка
    X_norm = l2norm(X)

    # 2) PCA фиксирован
    X_pca = PCA(n_components=31).fit_transform(X_norm)

    # 3) Подбираем параметры HDBSCAN
    min_cluster_size = trial.suggest_int("min_cluster_size", 5, 50)

    min_samples_mode = trial.suggest_categorical(
        "min_samples_mode", ["none", "value"]
    )
    if min_samples_mode == "none":
        min_samples = None
    else:
        min_samples = trial.suggest_int("min_samples", 1, 25)

    cluster_selection_method = trial.suggest_categorical(
        "cluster_selection_method", ["eom", "leaf"]
    )

    cluster_selection_epsilon = trial.suggest_float(
        "cluster_selection_epsilon", 0.0, 0.2
    )

    # 4) Усреднение по нескольким seed UMAP
    seeds = [0, 1, 2]
    scores = []

    for seed in seeds:
        reducer = umap.UMAP(
            n_components=10,
            n_neighbors=30,
            min_dist=0.0,
            metric="cosine",
            random_state=seed,
        )
        X_umap = reducer.fit_transform(X_pca)

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="euclidean",
            cluster_selection_method=cluster_selection_method,
            cluster_selection_epsilon=cluster_selection_epsilon,
            prediction_data=True,
        )

        clusters = clusterer.fit_predict(X_umap)

        noise_fraction = float(np.mean(clusters == -1))
        mask = clusters != -1

        # если все ушло в шум или остался 1 кластер — trial плохой
        if mask.sum() == 0 or len(np.unique(clusters[mask])) < 2:
            scores.append(-1.0)
            continue

        y_clean = np.asarray(y)[mask]
        clusters_clean = clusters[mask]

        macro_f1 = macro_f1_clustering(y_clean, clusters_clean)
        coverage = 1.0 - noise_fraction

        # честная цель: качество * покрытие
        objective_value = macro_f1 * coverage
        scores.append(objective_value)

    return float(np.mean(scores))

In [21]:

def l2norm(X, eps=1e-12):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.clip(norms, eps, None)

X, y = embeddings["osnet"]

sampler = optuna.samplers.TPESampler(seed=42)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

study.optimize(
    lambda trial: objective_hdbscan_pca31(trial, X, y),
    n_trials=80,
    show_progress_bar=True
)

print("best score:", study.best_value)
print("best params:", study.best_params)

[I 2026-03-11 09:05:06,456] A new study created in memory with name: no-name-91ba13ac-840f-4792-aff4-c86dc90dbfec


  0%|          | 0/80 [00:00<?, ?it/s]

d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 09:05:34,032] Trial 0 finished with value: 0.04203967365899582 and parameters: {'min_cluster_size': 22, 'min_samples_mode': 'none', 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.031198904067240532}. Best is trial 0 with value: 0.04203967365899582.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


...

[I 2026-03-11 09:33:25,483] Trial 77 finished with value: 0.3225251679714416 and parameters: {'min_cluster_size': 40, 'min_samples_mode': 'none', 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.10759392287675268}. Best is trial 12 with value: 0.571516688849048.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 09:33:48,913] Trial 78 finished with value: 0.5512520539367146 and parameters: {'min_cluster_size': 48, 'min_samples_mode': 'none', 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.12245234736543487}. Best is trial 12 with value: 0.571516688849048.


d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
d:\Anaconda\envs\footpass\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


[I 2026-03-11 09:34:11,078] Trial 79 finished with value: 0.1975399389601609 and parameters: {'min_cluster_size': 46, 'min_samples_mode': 'value', 'min_samples': 11, 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.10359749075226926}. Best is trial 12 with value: 0.571516688849048.
best score: 0.571516688849048
best params: {'min_cluster_size': 49, 'min_samples_mode': 'none', 'cluster_selection_method': 'eom', 'cluster_selection_epsilon': 0.07121824466699878}
